# Decompositions

Decomposing a matrix means writing it as simpler pieces. Eigendecomposition and SVD show which directions a matrix cares about, and least squares is where those ideas pay off in a real fit.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## 1. Intuition

Most vectors change direction when a matrix hits them. A few special ones only get stretched. Those are eigenvectors, and the stretch factor is the eigenvalue.

SVD works for any matrix, not just square ones. It says every matrix is a rotation, then an axis-aligned stretch, then another rotation.

## 2. Maths

Eigen: $Av = \lambda v$. For a symmetric $A$ (such as a covariance matrix) the eigenvectors are perpendicular and $A = Q \Lambda Q^T$.

SVD: $X = U \Sigma V^T$. $U$ and $V$ have orthonormal columns, $\Sigma$ is diagonal with singular values $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$.

Link: the singular values of $X$ squared are the eigenvalues of $X^T X$.

Low-rank approximation: keep the top $k$ singular values. This is the best rank-$k$ approximation in the least-squares sense (Eckart-Young).

Least squares: minimise $\lVert Xw - y \rVert^2$. Setting the gradient to zero gives the normal equation $X^T X w = X^T y$.

## 3. Cost

Full SVD of an $n \times d$ matrix costs about $O(n d^2)$. Keeping $k$ components stores $k(n + d + 1)$ numbers instead of $nd$. For a 1000 by 1000 image at $k = 20$ that is 40,020 numbers instead of 1,000,000, a 25x saving.

The normal equation needs $X^T X$, which costs $O(n d^2)$, then a solve at $O(d^3)$.

## 4. From scratch

### Eigenvectors by picture

A symmetric matrix applied to the unit circle gives an ellipse. The eigenvectors are the ellipse axes.

In [ ]:
A = np.array([[3.0, 1.0], [1.0, 3.0]])
vals, vecs = np.linalg.eigh(A)
print("eigenvalues:", vals)
print("eigenvectors (columns):\n", vecs)

theta = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(*circle, alpha=0.4, label="unit circle")
ax.plot(*(A @ circle), label="A applied")
for lam, v in zip(vals, vecs.T):
    ax.annotate("", xy=lam * v, xytext=(0, 0), arrowprops={"arrowstyle": "->", "color": "C3"})
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

v = vecs[:, 1]
print("A v == lambda v:", np.allclose(A @ v, vals[1] * v))

The ellipse axes have lengths 4 and 2, matching the eigenvalues.

### Power iteration

The dominant eigenvector falls out of repeated multiplication. Apply $A$, rescale, repeat.

In [ ]:
def power_iteration(A, steps=50, seed=0):
    v = np.random.default_rng(seed).normal(size=A.shape[0])
    for _ in range(steps):
        v = A @ v
        v /= np.linalg.norm(v)
    return v @ A @ v, v


lam, v = power_iteration(A)
print("dominant eigenvalue:", round(lam, 4), "vector:", v)

### SVD and low-rank approximation

Build a smooth 60 by 60 'image' plus a little noise. Rank 3 captures nearly all of it.

In [ ]:
x = np.linspace(0, 1, 60)
img = np.outer(np.sin(3 * x), np.cos(2 * x)) + 0.5 * np.outer(x, x**2)
img += 0.01 * rng.normal(size=img.shape)

U, s, Vt = np.linalg.svd(img, full_matrices=False)
print("top singular values:", s[:5])


def rank_k(U, s, Vt, k):
    return (U[:, :k] * s[:k]) @ Vt[:k, :]


for k in (1, 2, 3, 10):
    err = np.linalg.norm(img - rank_k(U, s, Vt, k)) / np.linalg.norm(img)
    print(f"rank {k:2d}: relative error {err:.4f}")

# check the link to eigenvalues of X^T X
eig = np.sort(np.linalg.eigvalsh(img.T @ img))[::-1]
print("sigma^2 == eig(X^T X):", np.allclose(s**2, eig))

Two components already give an error of about 2%, close to the noise floor. Real data behaves the same way: a few directions carry most of the signal.

### PCA pointer

PCA is the SVD of the centred data matrix. The right singular vectors are the principal components and $\sigma_i^2/(n-1)$ is the variance each explains. Section 04 builds it properly. Here is the one-line version:

In [ ]:
X = rng.normal(size=(200, 2)) @ np.array([[2.0, 0.0], [1.0, 0.5]])
Xc = X - X.mean(axis=0)
_, s_x, Vt_x = np.linalg.svd(Xc, full_matrices=False)
print("principal directions:\n", Vt_x)
print("variance explained:", s_x**2 / (s_x**2).sum())

### Least squares: the normal equation

Fit $y = 2 + 3x$ with noise. Add a column of ones so the intercept is just another weight.

In [ ]:
n = 100
x1 = rng.uniform(-1, 1, size=n)
X = np.column_stack([np.ones(n), x1])
y = 2.0 + 3.0 * x1 + 0.3 * rng.normal(size=n)

w_normal = np.linalg.solve(X.T @ X, X.T @ y)
print("normal equation:", w_normal)

# via SVD: w = V diag(1/s) U^T y
U, s, Vt = np.linalg.svd(X, full_matrices=False)
w_svd = Vt.T @ ((U.T @ y) / s)
print("SVD solution:   ", w_svd)

fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(x1, y, s=8, alpha=0.5)
xs = np.array([-1, 1])
ax.plot(xs, w_normal[0] + w_normal[1] * xs, "C1")
plt.show()

## 5. Library version

In [ ]:
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("np.linalg.lstsq:", w_lstsq)

lr = LinearRegression().fit(x1.reshape(-1, 1), y)
print("sklearn:", lr.intercept_, lr.coef_)
print(
    "all agree:",
    np.allclose(w_normal, w_lstsq) and np.allclose(w_lstsq, [lr.intercept_, *lr.coef_]),
)

## 6. What breaks

The normal equation squares the condition number. $\kappa(X^T X) = \kappa(X)^2$. When features are nearly collinear this hurts.

In [ ]:
x_a = rng.uniform(-1, 1, size=n)
X_bad = np.column_stack([x_a, x_a + 1e-6 * rng.normal(size=n)])
y_bad = x_a + 0.1 * rng.normal(size=n)

print("cond(X):    ", f"{np.linalg.cond(X_bad):.2e}")
print("cond(X^T X):", f"{np.linalg.cond(X_bad.T @ X_bad):.2e}")
print("normal eq weights:", np.linalg.solve(X_bad.T @ X_bad, X_bad.T @ y_bad))
print("lstsq weights:    ", np.linalg.lstsq(X_bad, y_bad, rcond=None)[0])

The two features are almost the same, so the data cannot say how to split the weight between them. The weights can be large and opposite in sign. A ridge penalty fixes this and appears in section 03. The library solver is the safer default because it works from the SVD and avoids squaring the condition number.